# 03 — Parquet, Iceberg & Delta Lake

In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()



## Parquet + Partition Pruning

In [ ]:
schema = StructType([
    StructField('id',IntegerType()), StructField('name',StringType()),
    StructField('department',StringType()), StructField('salary',DoubleType()),
    StructField('country',StringType()),
])
employees = spark.createDataFrame([
    (1,'Alice','Engineering',95000.0,'US'),(2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),(4,'Dave','HR',65000.0,'DE'),
], schema)
employees.write.mode('overwrite').partitionBy('country').parquet(f'{DATA_DIR}/employees_parquet')

t0 = time.time()
us_only = spark.read.parquet(f'{DATA_DIR}/employees_parquet').filter(F.col('country')=='US')
print('US rows:', us_only.count(), f'  {time.time()-t0:.2f}s')
us_only.explain()


## Apache Iceberg — ACID · Time Travel · Schema Evolution

In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.demo')
employees.writeTo('local.demo.employees_ice').createOrReplace()
spark.sql("UPDATE local.demo.employees_ice SET salary = salary * 1.1 WHERE department='Engineering'")
spark.sql('SELECT * FROM local.demo.employees_ice').show()


In [ ]:
# Time travel
history = spark.sql('SELECT snapshot_id, committed_at, operation FROM local.demo.employees_ice.snapshots ORDER BY committed_at')
history.show(truncate=False)
first_snap = history.first()['snapshot_id']
spark.read.option('snapshot-id', first_snap).table('local.demo.employees_ice').show()


In [ ]:
# Schema evolution
spark.sql("ALTER TABLE local.demo.employees_ice ADD COLUMN bonus DOUBLE")
spark.sql("UPDATE local.demo.employees_ice SET bonus = salary * 0.1")
spark.sql('SELECT name, salary, bonus FROM local.demo.employees_ice').show()


## Delta Lake — ACID · Time Travel · OPTIMIZE

In [ ]:
employees.write.format('delta').mode('overwrite').partitionBy('country').save(f'{DATA_DIR}/employees_delta')
spark.sql(f"CREATE TABLE IF NOT EXISTS employees_delta USING delta LOCATION '{DATA_DIR}/employees_delta'")
spark.sql("UPDATE employees_delta SET salary = salary * 1.2 WHERE country='US'")
spark.sql('SELECT * FROM employees_delta').show()


In [ ]:
# Time travel
spark.read.format('delta').option('versionAsOf', 0).load(f'{DATA_DIR}/employees_delta').show()
spark.sql('DESCRIBE HISTORY employees_delta').select('version','timestamp','operation').show()
